In [1]:
import os
import numpy as np
import pandas as pd
import tifffile as tiff


In [5]:
### Variables
DATA = '../data/full_size_test/30.tif'
MODEL1 = 'paolo_model2'
SAVE_DATA_DIR = f'../masks/{MODEL1}_trackastra_test'
ERK_MASK ='erk_mask_c3_erk.npz'


imgs = tiff.imread('/mnt/imaging.data/PertzLab/apoDetection/TIFFs/Exp07_Site01.tif')



erk_mask_path = '/home/nbahou/myimaging/apoDet/data/dataset1/apo_masks/Exp07_Site01.npz'
# os.path.join(SAVE_DATA_DIR, ERK_MASK)


masks = np.load(erk_mask_path)['gt']

In [8]:
print(imgs.shape)


(1930, 1024, 1024)


In [ ]:
import torch
from trackastra.model import Trackastra
from trackastra.tracking import graph_to_ctc, graph_to_napari_tracks
from trackastra.data import example_data_bacteria

device = "cuda" if torch.cuda.is_available() else "cpu"

# load some test data images and masks
# imgs, masks = erk_imgs, masks

# Load a pretrained model
model = Trackastra.from_pretrained("general_2d", device=device)

# or from a local folder
# model = Trackastra.from_folder('path/my_model_folder/', device=device)

# Track the cells
track_graph = model.track(imgs, masks, mode="greedy")  # or mode="ilp", or "greedy_nodiv"


# Write to cell tracking challenge format
ctc_tracks, masks_tracked = graph_to_ctc(
      track_graph,
      masks,
      outdir="tracked",
)

/home/nbahou/.trackastra/.models/general_2d already downloaded, skipping.


INFO:trackastra.model.model:Loading model state from /home/nbahou/.trackastra/.models/general_2d/model.pt
INFO:trackastra.model.model_api:Using device cuda
INFO:trackastra.model.model_api:Predicting weights for candidate graph
INFO:trackastra.data.wrfeat:Extracting features from 1930 detections
INFO:trackastra.data.wrfeat:Using single process for feature extraction
Extracting features: 100%|██████████████████████████████████████████████████████████| 1930/1930 [11:52<00:00,  2.71it/s]
INFO:trackastra.model.model_api:Building windows
Building windows: 100%|███████████████████████████████████████████████████████████| 1927/1927 [00:00<00:00, 3286.59it/s]
INFO:trackastra.model.model_api:Predicting windows
Computing associations:  66%|█████████████████████████████████▋                 | 1274/1927 [1:24:43<2:40:21, 14.73s/it]

In [5]:
# Visualise in napari
napari_tracks, napari_tracks_graph, _ = graph_to_napari_tracks(track_graph)

print(napari_tracks_graph)

# Save both the track data and the graph in a compressed .npz file
#np.savez_compressed("../data/trackastra_tracks.npz", 
#                    napari_tracks=napari_tracks, 
#                    napari_tracks_graph=napari_tracks_graph)

#import napari
#v = napari.Viewer()
#v.add_image(imgs)
#v.add_labels(masks_tracked)
#v.add_tracks(data=napari_tracks, graph=napari_tracks_graph)

100%|██████████████████████████████████████████████████████████████████████████| 22176/22176 [00:00<00:00, 42864.92it/s]

{1: 8478, 2: 8478, 3: 816, 4: 816, 10: 909, 11: 909, 21: 6699, 22: 6699, 23: 1986, 24: 1986, 25: 2489, 26: 2489, 31: 16609, 32: 16609, 33: 9187, 34: 9187, 38: 14193, 39: 14193, 41: 7091, 42: 7091, 47: 5895, 48: 5895, 50: 475, 51: 475, 54: 4597, 55: 4597, 56: 312, 57: 312, 61: 6317, 62: 6317, 64: 5878, 65: 5878, 66: 4271, 67: 4271, 74: 8723, 75: 8723, 77: 2869, 78: 2869, 80: 8670, 81: 8670, 85: 828, 86: 828, 91: 2924, 92: 2924, 96: 3047, 97: 3047, 99: 895, 100: 895, 106: 10342, 107: 10342, 109: 21211, 110: 21211, 113: 19249, 114: 19249, 116: 2832, 117: 2832, 124: 16998, 125: 16998, 136: 3889, 137: 3889, 139: 18174, 140: 18174, 142: 20299, 143: 20299, 151: 9758, 152: 9758, 164: 8442, 165: 8442, 166: 22103, 167: 22103, 176: 7440, 177: 7440, 178: 2195, 179: 2195, 195: 19574, 196: 19574, 198: 238, 199: 238, 202: 868, 203: 868, 205: 4051, 206: 4051, 212: 8435, 213: 8435, 214: 2521, 215: 2521, 221: 22020, 222: 22020, 223: 4726, 224: 4726, 225: 406, 226: 406, 229: 21993, 230: 21993, 241: 14199

In [ ]:
napari_track